In [9]:
import os
import gc
import numpy as np
import pandas as pd

# 1. Path to raw data
raw_bb_path = '/Users/nguyenminhtri/FinalYearPro/data/raw/bureau_balance.csv'

print(f"📥 Loading raw dataset from: '{raw_bb_path}'...")
df_bb = pd.read_csv(raw_bb_path)

print("=" * 60)
print(f"📊 Raw Dataset Shape: {df_bb.shape[0]:,} rows | {df_bb.shape[1]} columns")
print("=" * 60)
print("📋 First 5 rows:")
print(df_bb.head())
print("\n📋 Missing Values Audit:")
print(df_bb.isnull().sum())
print("=" * 60)

📥 Loading raw dataset from: '/Users/nguyenminhtri/FinalYearPro/data/raw/bureau_balance.csv'...
📊 Raw Dataset Shape: 27,299,925 rows | 3 columns
📋 First 5 rows:
   SK_ID_BUREAU  MONTHS_BALANCE STATUS
0       5715448               0      C
1       5715448              -1      C
2       5715448              -2      C
3       5715448              -3      C
4       5715448              -4      C

📋 Missing Values Audit:
SK_ID_BUREAU      0
MONTHS_BALANCE    0
STATUS            0
dtype: int64


In [10]:
# --- STEP 3: ROW-LEVEL FEATURE ENGINEERING ---

print("⚙️ Generating row-level features for 27.3M records...")

# 1. One-Hot Encoding for STATUS
status_dummies = pd.get_dummies(df_bb['STATUS'], prefix='BB_STATUS').astype(np.int8)
df_bb = pd.concat([df_bb, status_dummies], axis=1)

# 2. Risk Buckets
df_bb['BB_IS_OVERDUE'] = df_bb['STATUS'].isin(['0', '1', '2', '3', '4', '5']).astype(np.int8)
df_bb['BB_IS_DPD_30PLUS'] = df_bb['STATUS'].isin(['1', '2', '3', '4', '5']).astype(np.int8)
df_bb['BB_IS_NPL_90PLUS'] = df_bb['STATUS'].isin(['3', '4', '5']).astype(np.int8)

# 3. Recency Time Windows (Flags)
df_bb['BB_IN_RECENT_6M'] = (df_bb['MONTHS_BALANCE'] >= -6).astype(np.int8)
df_bb['BB_IN_RECENT_12M'] = (df_bb['MONTHS_BALANCE'] >= -12).astype(np.int8)
df_bb['BB_IN_RECENT_24M'] = (df_bb['MONTHS_BALANCE'] >= -24).astype(np.int8)

print("=" * 60)
print("🎉 COMPLETED ROW-LEVEL FEATURE ENGINEERING!")
print("=" * 60)
print(f"📊 Dataset Shape: {df_bb.shape[0]:,} rows | {df_bb.shape[1]} columns")
print("\n📋 Sample Created Columns:")
print(list(df_bb.columns[-10:]))
print("=" * 60)

⚙️ Generating row-level features for 27.3M records...
🎉 COMPLETED ROW-LEVEL FEATURE ENGINEERING!
📊 Dataset Shape: 27,299,925 rows | 17 columns

📋 Sample Created Columns:
['BB_STATUS_4', 'BB_STATUS_5', 'BB_STATUS_C', 'BB_STATUS_X', 'BB_IS_OVERDUE', 'BB_IS_DPD_30PLUS', 'BB_IS_NPL_90PLUS', 'BB_IN_RECENT_6M', 'BB_IN_RECENT_12M', 'BB_IN_RECENT_24M']


## 📌 Step 3: Feature Engineering (Row-Level Transformations)
- **One-Hot Encoding:** Convert categorical `STATUS` into 8 binary indicators (`STATUS_C`, `STATUS_0`, etc.).
- **Risk Buckets:** Create regulatory delinquency buckets (`DPD_30PLUS`, `NPL_90PLUS`).
- **Recency Windowing:** Flag entries belonging to recent time windows (6M, 12M, 24M) to capture trend changes.

In [11]:
# --- ADVANCED TIME SERIES FEATURE ENGINEERING (STEP 4) ---

print("⚡ Generating Advanced Time Series Features (Severity, Recency, Dynamics)...")

# 1. Map STATUS to Numerical Severity Score (Thang điểm nghiêm trọng 0 - 6)
severity_map = {'C': 0, 'X': 0, '0': 1, '1': 2, '2': 3, '3': 4, '4': 5, '5': 6}
df_bb['BB_SEVERITY_SCORE'] = df_bb['STATUS'].map(severity_map).fillna(0).astype(np.int8)

# 2. Distance in months for overdue records (Chỉ lấy tháng bị trễ nợ)
# Nếu không trễ nợ thì để NaN, tí nữa lấy MAX của MONTHS_BALANCE sẽ ra tháng trễ nợ gần nhất
df_bb['BB_OVERDUE_MONTHS_BALANCE'] = np.where(df_bb['BB_IS_OVERDUE'] == 1, df_bb['MONTHS_BALANCE'], np.nan)

print("✅ Advanced Time Series Features prepared successfully!")
print(f"📊 Dataset Shape: {df_bb.shape[0]:,} rows | {df_bb.shape[1]} columns")

⚡ Generating Advanced Time Series Features (Severity, Recency, Dynamics)...
✅ Advanced Time Series Features prepared successfully!
📊 Dataset Shape: 27,299,925 rows | 19 columns


## 📌 Step 3.5: Advanced Time Series Transformation (Severity & Recency Engineering)

### 💡 Justification & Domain Rationale:
- **Numerical Severity Mapping (`BB_SEVERITY_SCORE`):** Converts non-numeric string status codes (`C`, `0`, `X`, `1`–`5`) into a ordinal risk hierarchy scale ($0 \rightarrow 6$). This enables mathematical aggregation (e.g., `MAX` peak severity, `MEAN` average stress) across historical months.
  - `C` / `X` $\rightarrow 0$ (Zero risk / Neutral)
  - `0` $\rightarrow 1$ (Minor delay / Normal)
  - `1` $\rightarrow 2$ (30–59 days overdue - Caution)
  - `2` $\rightarrow 3$ (60–89 days overdue)
  - `3` $\rightarrow 4$ (90–119 days overdue - Substandard)
  - `4` $\rightarrow 5$ (120–179 days overdue)
  - `5` $\rightarrow 6$ (180+ days / Severe Bad Debt - NPL)
- **Recency Distance Tracking (`BB_OVERDUE_MONTHS_BALANCE`):** Isolates exact `MONTHS_BALANCE` locations where delinquency occurred. Taking the `MAX` during aggregation yields the **most recent month of default** relative to the application date.

In [12]:
import numpy as np
import pandas as pd

print("🚀 Đang tiến hành Groupby Level 1 nén 27.3M dòng về từng khoản vay (SK_ID_BUREAU)...")

# 1. KHAI BÁO DICTIONARY AGGREGATION LEVEL 1
bb_agg_dict = {
    # Mốc thời gian & độ dài lịch sử
    'MONTHS_BALANCE': ['min', 'size'],

    # Thang điểm nghiêm trọng
    'BB_SEVERITY_SCORE': ['max', 'mean'],

    # Mốc tháng trễ nợ gần nhất (dùng max vì số âm, max nghĩa là gần 0 nhất)
    'BB_OVERDUE_MONTHS_BALANCE': ['max'],

    # Tần suất xuất hiện các trạng thái OHE
    'BB_STATUS_0': ['sum', 'mean'],
    'BB_STATUS_1': ['sum', 'mean'],
    'BB_STATUS_2': ['sum', 'mean'],
    'BB_STATUS_3': ['sum', 'mean'],
    'BB_STATUS_4': ['sum', 'mean'],
    'BB_STATUS_5': ['sum', 'mean'],
    'BB_STATUS_C': ['sum', 'mean'],
    'BB_STATUS_X': ['sum', 'mean'],

    # Cờ rủi ro tổng hợp
    'BB_IS_OVERDUE': ['sum', 'mean'],
    'BB_IS_DPD_30PLUS': ['sum', 'mean'],
    'BB_IS_NPL_90PLUS': ['sum', 'mean'],

    # Cửa sổ thời gian
    'BB_IN_RECENT_6M': ['sum'],
    'BB_IN_RECENT_12M': ['sum'],
    'BB_IN_RECENT_24M': ['sum']
}

# Lọc lấy các cột thực sự có trong df_bb
valid_bb_aggs = {col: aggs for col, aggs in bb_agg_dict.items() if col in df_bb.columns}

# 2. GROUPBY THEO SK_ID_BUREAU
df_bb_agg = df_bb.groupby('SK_ID_BUREAU').agg(valid_bb_aggs)

# Ép phẳng tên cột
df_bb_agg.columns = [f"{col}_{stat.upper()}" for col, stat in df_bb_agg.columns]
df_bb_agg.reset_index(inplace=True)

# 3. TÍNH TOÁN CÁC CHỈ SỐ TIME SERIES NÂNG CAO (TREND & RECENCY)

# A. Khoảng cách thời gian (số tháng) từ lần trễ nợ gần nhất tới hiện tại
# Do MONTHS_BALANCE âm, ta lấy giá trị tuyệt đối |max|
df_bb_agg['BB_MONTHS_SINCE_LAST_OVERDUE'] = df_bb_agg['BB_OVERDUE_MONTHS_BALANCE_MAX'].abs()
df_bb_agg.drop(columns=['BB_OVERDUE_MONTHS_BALANCE_MAX'], inplace=True)

# B. Chỉ số Trend (Momentum): Tỷ lệ nợ xấu bùng phát trong 6 tháng gần nhất
# Công thức: (Số tháng overdue trong 6M) / (Tổng số tháng overdue trong lịch sử + eps)
if 'BB_IS_OVERDUE_SUM' in df_bb_agg.columns:
    # Lọc số tháng overdue nằm trong 6M gần nhất từ df_bb
    recent_6m_overdue = df_bb[df_bb['MONTHS_BALANCE'] >= -6].groupby('SK_ID_BUREAU')['BB_IS_OVERDUE'].sum().reset_index()
    recent_6m_overdue.columns = ['SK_ID_BUREAU', 'BB_OVERDUE_SUM_RECENT_6M']

    df_bb_agg = df_bb_agg.merge(recent_6m_overdue, on='SK_ID_BUREAU', how='left')
    df_bb_agg['BB_OVERDUE_SUM_RECENT_6M'] = df_bb_agg['BB_OVERDUE_SUM_RECENT_6M'].fillna(0)

    # Trend ratio: Tỷ lệ tập trung nợ xấu ở quá khứ gần
    df_bb_agg['BB_OVERDUE_TREND_RATIO_6M'] = (
        df_bb_agg['BB_OVERDUE_SUM_RECENT_6M'] / (df_bb_agg['BB_IS_OVERDUE_SUM'] + 1e-5)
    )

print("=" * 60)
print("🎉 HOÀN THÀNH GROUPBY LEVEL 1 (SK_ID_BUREAU) & TIME SERIES DYNAMICS!")
print("=" * 60)
print(f"📊 Kích thước bảng nén: {df_bb_agg.shape[0]:,} khoản vay unique | {df_bb_agg.shape[1]} thuộc tính")
print("\n📋 Mẫu 10 tên cột thuộc tính vừa tạo:")
for col in list(df_bb_agg.columns)[1:11]:
    print(f"   • {col}")
print("=" * 60)

🚀 Đang tiến hành Groupby Level 1 nén 27.3M dòng về từng khoản vay (SK_ID_BUREAU)...
🎉 HOÀN THÀNH GROUPBY LEVEL 1 (SK_ID_BUREAU) & TIME SERIES DYNAMICS!
📊 Kích thước bảng nén: 817,395 khoản vay unique | 33 thuộc tính

📋 Mẫu 10 tên cột thuộc tính vừa tạo:
   • MONTHS_BALANCE_MIN
   • MONTHS_BALANCE_SIZE
   • BB_SEVERITY_SCORE_MAX
   • BB_SEVERITY_SCORE_MEAN
   • BB_STATUS_0_SUM
   • BB_STATUS_0_MEAN
   • BB_STATUS_1_SUM
   • BB_STATUS_1_MEAN
   • BB_STATUS_2_SUM
   • BB_STATUS_2_MEAN


## 📌 Step 4: Level 1 Aggregation per Loan (`SK_ID_BUREAU`) & Trend Dynamics

### 💡 Justification & Strategy:
- Compress 27.3M monthly records into unique loan-level features (`SK_ID_BUREAU`).
- Extract time-series momentum indicators (Recent 6M vs. Historical overdue ratio).
- Compute recency metrics (Months elapsed since the last default event).

In [13]:
import os
import gc

# 1. Đường dẫn lưu file bureau_balance đã FE (ở cấp SK_ID_BUREAU)
output_path = '/data/processed/table/df_bureau_balance_clean_fe.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# 2. Lưu trực tiếp dataframe df_bb_agg (đã nén ở Step 4)
df_bb_agg.to_parquet(output_path, index=False)

print("=" * 60)
print("🎉 THÀNH CÔNG: ĐÃ LƯU FILE BUREAU BALANCE XỬ LÝ ĐỘC LẬP!")
print("=" * 60)
print(f"📊 Shape (Cấp Khoản vay SK_ID_BUREAU) : {df_bb_agg.shape[0]:,} dòng | {df_bb_agg.shape[1]} cột")
print(f"💾 File Path                         : {output_path}")
print("=" * 60)

# 3. Dọn dẹp RAM sạch sẽ
del df_bb, df_bb_agg
gc.collect()
print("🧹 Memory cleared successfully! Notebook 03 ĐÃ HOÀN THÀNH CHUẨN ĐỘC LẬP.")

🎉 THÀNH CÔNG: ĐÃ LƯU FILE BUREAU BALANCE XỬ LÝ ĐỘC LẬP!
📊 Shape (Cấp Khoản vay SK_ID_BUREAU) : 817,395 dòng | 33 cột
💾 File Path                         : /Users/nguyenminhtri/FinalYearPro/data/processed/df_bureau_balance_clean_fe.parquet
🧹 Memory cleared successfully! Notebook 03 ĐÃ HOÀN THÀNH CHUẨN ĐỘC LẬP.
